# ❄️ 第一周：AI 的初春与寒冬 —— 感知机的诞生与 XOR 危机

非常理解你的需求！这一次，我们将采用**“步步为营、边讲边写”**的方式。我们不再把代码全堆在最后，而是讲透一个概念，就立刻用一行行代码把它具象化。

在 20 世纪 50 年代，科学家们对人类大脑产生了极大的好奇。大脑里有 860 亿个神经元，它们是如何产生智能的？
心理学家弗兰克·罗森布拉特（Frank Rosenblatt）在 1957 年提出了一个极简的数学模型——**感知机（Perceptron）**。这就是今天所有人工智能、包括 ChatGPT 的祖师爷。

---

## 1. 核心概念：什么是感知机？

感知机的灵感完全来源于生物神经元。
在生物学中，一个神经元会接收来自其他多个神经元的电信号刺激。如果这些刺激的总和足够强，超过了某个“阈值”，这个神经元就会“兴奋”，并把信号传给下一个神经元；如果没有达到阈值，它就保持“沉默”。

**在数学世界里，我们怎么模拟这个过程呢？**
我们需要一个容器来接收数据，并且需要一种机制来模拟“刺激总和”以及“是否达到阈值”。

首先，我们用代码把这个“空壳容器”建出来。我们使用 PyTorch 的核心数据结构——**张量（Tensor）** 来作为我们运算的基础。张量就像是多维数组，是深度学习世界里的“通用货币”。

In [ ]:
import torch

# 我们定义一个名叫 Perceptron (感知机) 的类，用来模拟生物神经元
class Perceptron:
    def __init__(self, input_features_count):
        """
        初始化感知机
        :param input_features_count: 接收多少个外界输入信号（也就是树突的数量）
        """
        self.input_features_count = input_features_count
        
        # 此时感知机还是个空壳，我们还没有给它赋予“学习”的能力
        print(f"✅ 成功创建了一个感知机，它可以同时接收 {self.input_features_count} 个输入信号。")

# 实例化一个感知机，假设我们要让它做决定：今天是否要去打球？
# 输入 1：天气是否晴朗？ (1=晴, 0=雨)
# 输入 2：朋友是否叫我？ (1=叫, 0=没叫)
my_neuron = Perceptron(input_features_count=2)


---

## 2. 核心概念：权重（Weight）与偏置（Bias）

光有一个空壳是不够的。不同的输入信号，对你做决定的“影响力”是完全不同的。

**什么是权重（Weight）？**
假设你非常热爱篮球，只要天气晴朗（输入1），你大概率就会去打球；而朋友叫不叫你（输入2），只是个锦上添花的因素。
在数学上，我们给每个输入信号分配一个乘数，这就是**权重 (Weight)**。
- 天气的权重 $W_1$ 可能很大，比如 5.0。
- 朋友的权重 $W_2$ 可能很小，比如 1.0。
权重代表了某个特征的**重要程度**。在神经网络中，**“学习”的本质，其实就是不断调整这些权重的值，直到找出一组完美的组合。**

**什么是偏置（Bias）？**
假设你是一个极度死宅的人，哪怕天气再好、朋友再怎么叫，你都不太想出门。这种与生俱来的“基础门槛”或者“天生倾向”，就是**偏置 (Bias)**。
偏置独立于外界输入，它直接决定了这个神经元有多么**“容易被激活”**。一个负数很大的偏置，意味着这个神经元极其高冷，需要外界输入非常强烈的信号才能把它唤醒。

我们现在把权重和偏置加入到我们的代码中。

In [ ]:
import torch
class PerceptronWithMemory(Perceptron):
    def __init__(self, input_features_count):
        super().__init__(input_features_count)
        
        # 1. 初始化权重 (Weight)
        # 因为有 input_features_count 个输入，所以我们需要对应数量的权重。
        # 我们生成一个形状为 [input_features_count, 1] 的列向量张量。
        # 初始状态下，机器是一张白纸，所以我们用 torch.randn 赋予它随机的初始值。
        self.weights = torch.randn(input_features_count, 1)
        
        # 2. 初始化偏置 (Bias)
        # 偏置只有一个值，代表神经元整体的激活门槛。同样随机初始化。
        self.bias = torch.randn(1)
        
        print("🧠 神经元获得了权重和偏置！当前的初始状态是：")
        print(f"权重 W:\n{self.weights}")
        print(f"偏置 B: {self.bias.item():.4f}")

# 重新实例化我们的神经元，这次它有了随机的记忆
my_neuron = PerceptronWithMemory(input_features_count=2)


---

## 3. 核心概念：线性分类边界与激活函数

有了输入 $X$、权重 $W$ 和偏置 $Bias$，神经元内部是怎么计算的呢？

**第一步：加权求和（线性组合）**
神经元会把收到的信号，乘以它们各自的权重，再加上偏置。
公式：$Z = (X_1 \times W_1) + (X_2 \times W_2) + Bias$
在二维空间中，如果我们令 $Z = 0$，这其实就是初中数学里**一条直线的方程**（类似 $y = kx + b$）。
这条直线，就是感知机的**“线性分类边界”**。它试图用这条直线，把你输入的数据一刀切成两半（一半是通过，一半是不通过）。

**第二步：激活函数（Activation Function）**
算出了总得分 $Z$ 后，神经元要做出最终的决定：输出 1（激活）还是 0（沉默）。
早期的感知机使用了一种简单粗暴的**“阶跃函数 (Step Function)”**：
- 如果 $Z > 0$，输出 1。
- 如果 $Z \le 0$，输出 0。

让我们在代码中实现这个“思考”的过程，也就是神经网络大名鼎鼎的**前向传播（Forward Pass）**。

In [ ]:
import torch

class Perceptron:
    """感知机骨架"""
    def __init__(self, input_features_count):
        self.input_features_count = input_features_count

class PerceptronWithMemory(Perceptron):
    """带权重和偏置的感知机"""
    def __init__(self, input_features_count):
        super().__init__(input_features_count)
        self.weights = torch.randn(input_features_count, 1)
        self.bias = torch.randn(1)
    
    def forward(self, inputs):
        z = torch.matmul(inputs, self.weights) + self.bias
        return (z > 0).float()

import torch
class FullyFunctionalPerceptron(PerceptronWithMemory):
    
    def forward(self, inputs):
        """
        前向传播：感知机处理信息的全过程
        :param inputs: 外界给到的输入数据矩阵
        """
        # 1. 加权求和 (线性分类边界的基础)
        # inputs 的形状通常是 [样本数, 特征数]，weights 的形状是 [特征数, 1]
        # torch.matmul() 是矩阵乘法，它能瞬间完成每个特征与权重的相乘并求和
        # 最后加上 bias 偏置。
        # 这个 linear_output 就是总得分 Z
        linear_output = torch.matmul(inputs, self.weights) + self.bias
        
        # 2. 激活函数 (阶跃函数)
        # 判断得分是否大于 0。大于0返回 True(变浮点数就是1.0)，否则返回 False(0.0)
        final_decision = (linear_output > 0).float()
        
        return final_decision

# 让我们测试一下！
# 假设我们强行赋予神经元一组非常明确的“价值观 (权重)”
my_neuron = FullyFunctionalPerceptron(input_features_count=2)
my_neuron.weights = torch.tensor([[5.0],  # 天气的权重很大
                                  [1.0]]) # 朋友的权重较小
my_neuron.bias = torch.tensor([-4.0])     # 偏置为负，人比较宅，初始门槛较高

print("=== 测试打篮球神经元 ===")
# 情境 1：天气不好(0)，朋友叫了(1)
situation_1 = torch.tensor([[0.0, 1.0]])
decision_1 = my_neuron.forward(situation_1)
# 解释: (0*5.0) + (1*1.0) - 4.0 = -3.0 <= 0, 所以输出 0
print(f"情境1 (雨天, 朋友叫): 得分未超阈值，输出 -> {decision_1.item()} (不去)")

# 情境 2：天气晴朗(1)，朋友没叫(0)
situation_2 = torch.tensor([[1.0, 0.0]])
decision_2 = my_neuron.forward(situation_2)
# 解释: (1*5.0) + (0*1.0) - 4.0 = 1.0 > 0, 所以输出 1
print(f"情境2 (晴天, 朋友没叫): 得分超过阈值，输出 -> {decision_2.item()} (去打球！)")


---

## 4. 历史的崩塌：XOR 危机

感知机虽然能解决像上面“去不去打球”这样简单的逻辑问题，但它有一个致命的几何缺陷。

**缺陷在于：它的决策本质是在画一条“直线”。**
只要是单层感知机，它就只能解决**“线性可分”**的数据（也就是能用一条直线把黑白两派切开的数据）。

1969 年，人工智能领域的巨擘明斯基（Minsky）提出了**异或（XOR）问题**。
异或的逻辑是：**两个输入不同时为真（1），相同时为假（0）。**
我们来看看 XOR 的数据点分布：

```text
       X2 轴
        |
    1   |   (0,1) [结果:1]        (1,1) [结果:0]
        |      🟢 去打球                🔴 不去
        |
        |
    0   |   (0,0) [结果:0]        (1,0) [结果:1]
        |      🔴 不去                  🟢 去打球
        |____________________________________ X1 轴
            0                     1
```

请你尝试在这个二维平面上，**画出一条绝对笔直的直线**，把 🟢 和 🔴 完美地分开。
你会发现：**根本不可能！** 无论你怎么画，都会有一边混入错误颜色的点。因为它们是对角交叉的。

这就是导致 AI 历史上第一次十年寒冬的根本原因：**人类寄予厚望的感知机模型，居然连这么基础的非线性逻辑都无法表达！**

下面，我们用代码亲自验证这场历史性的挫败。

In [ ]:
import torch

class Perceptron:
    """感知机骨架"""
    def __init__(self, input_features_count):
        self.input_features_count = input_features_count

class PerceptronWithMemory(Perceptron):
    """带权重和偏置的感知机"""
    def __init__(self, input_features_count):
        super().__init__(input_features_count)
        self.weights = torch.randn(input_features_count, 1)
        self.bias = torch.randn(1)
    def forward(self, inputs):
        z = torch.matmul(inputs, self.weights) + self.bias
        return (z > 0).float()

# FullyFunctionalPerceptron 类定义（自包含）
import torch
class FullyFunctionalPerceptron(PerceptronWithMemory):
    
    def forward(self, inputs):
        """
        前向传播：感知机处理信息的全过程
        :param inputs: 外界给到的输入数据矩阵
        """
        # 1. 加权求和 (线性分类边界的基础)
        # inputs 的形状通常是 [样本数, 特征数]，weights 的形状是 [特征数, 1]
        # torch.matmul() 是矩阵乘法，它能瞬间完成每个特征与权重的相乘并求和
        # 最后加上 bias 偏置。
        # 这个 linear_output 就是总得分 Z
        linear_output = torch.matmul(inputs, self.weights) + self.bias
        
        # 2. 激活函数 (阶跃函数)
        # 判断得分是否大于 0。大于0返回 True(变浮点数就是1.0)，否则返回 False(0.0)
        final_decision = (linear_output > 0).float()
        
        return final_decision

# 让我们测试一下！
# 假设我们强行赋予神经元一组非常明确的“价值观 (权重)”
my_neuron = FullyFunctionalPerceptron(input_features_count=2)
my_neuron.weights = torch.tensor([[5.0],  # 天气的权重很大
                                  [1.0]]) # 朋友的权重较小
my_neuron.bias = torch.tensor([-4.0])     # 偏置为负，人比较宅，初始门槛较高

print("=== 测试打篮球神经元 ===")
# 情境 1：天气不好(0)，朋友叫了(1)
situation_1 = torch.tensor([[0.0, 1.0]])
decision_1 = my_neuron.forward(situation_1)
# 解释: (0*5.0) + (1*1.0) - 4.0 = -3.0 <= 0, 所以输出 0
print(f"情境1 (雨天, 朋友叫): 得分未超阈值，输出 -> {decision_1.item()} (不去)")

# 情境 2：天气晴朗(1)，朋友没叫(0)
situation_2 = torch.tensor([[1.0, 0.0]])
decision_2 = my_neuron.forward(situation_2)
# 解释: (1*5.0) + (0*1.0) - 4.0 = 1.0 > 0, 所以输出 1
print(f"情境2 (晴天, 朋友没叫): 得分超过阈值，输出 -> {decision_2.item()} (去打球！)")

import torch
# 验证 XOR 危机
print("=== 面对 XOR 危机，感知机的绝望 ===")

# XOR 的四种输入组合 (X1, X2)
X_xor = torch.tensor([[0.0, 0.0],
                      [0.0, 1.0],
                      [1.0, 0.0],
                      [1.0, 1.0]])

# XOR 应该输出的正确结果
y_xor_true = torch.tensor([[0.0], [1.0], [1.0], [0.0]])

# 我们创建一个全新的感知机
doomed_neuron = FullyFunctionalPerceptron(input_features_count=2)

# 无论我们怎么疯狂地尝试调整它的权重和偏置
# (这里我们模拟模型学习了很多次，尝试了3组截然不同的参数)
for attempt in range(1, 4):
    print(f"\n--- 第 {attempt} 次调整参数尝试 ---")
    doomed_neuron.weights = torch.randn(2, 1)
    doomed_neuron.bias = torch.randn(1)
    
    # 获取预测结果
    predictions = doomed_neuron.forward(X_xor)
    
    # 逐个对比
    all_correct = True
    for i in range(4):
        x_val = X_xor[i].tolist()
        true_val = int(y_xor_true[i].item())
        pred_val = int(predictions[i].item())
        
        if true_val == pred_val:
            print(f"输入 {x_val} -> 预测 {pred_val} (正确 ✅)")
        else:
            print(f"输入 {x_val} -> 预测 {pred_val} (错误 ❌) [真理应为 {true_val}]")
            all_correct = False
            
    if all_correct:
        print("竟然全对？！(这在数学上是不可能的，如果你看到了说明宇宙出bug了)")
    else:
        print("结论：未能全部预测正确。感知机画出的一根直线，终究切不开 XOR 的鸿沟。")


---
### 总结与预告

在本周，我们一步步敲出了感知机的代码，了解了：
1. **张量运算** 是神经网络处理信息的基础。
2. **权重 (Weight)** 衡量特征的重要性；**偏置 (Bias)** 决定激活的基础门槛。
3. **前向传播** 就是 `Z = X*W + Bias` 加上阶跃函数的判断，这在几何上等价于画一条直线（**线性分类边界**）。
4. **XOR 危机** 证明了单层网络“一条路走到黑”的局限性。

既然一条直线切不开，下周我们将寻找破局之道：如果把空间扭曲一下（非线性激活），或者画多条折线（加入隐藏层），是不是就能解决了？
敬请期待第二周：**破局与复兴 —— 隐藏层与反向传播的奇迹**。